---
## 3. Data Inspection & Cleaning

Before any analysis, we inspect the raw data, understand its structure, and apply a reproducible cleaning pipeline.

---
## 0. Load Retrieved FIRMS Data

This cleaning notebook continues from `data_retrieval.ipynb` by loading the cached raw CSV written to `data/raw/`. This makes the notebook reproducible even when it is run in a fresh kernel.


In [2]:
# --- Standard library ---
from pathlib import Path
import warnings

# --- Data manipulation ---
import numpy as np
import pandas as pd

# --- Geospatial ---
import geopandas as gpd

warnings.filterwarnings("ignore", category=FutureWarning)

# Robust project-root detection:
# If this notebook is run from a notebooks/ folder, the project root is one level up.
# If it is run from the project root, the project root is the current folder.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebook", "notebooks"} else cwd

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# This file should have been created by data_retrieval.ipynb
RAW_CSV = RAW_DATA_DIR / "firms_viirs_global_5d.csv"

# Fallback: if the exact filename is different, use the newest FIRMS CSV in data/raw.
if not RAW_CSV.exists():
    candidates = sorted(RAW_DATA_DIR.glob("firms*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        RAW_CSV = candidates[0]
    else:
        raise FileNotFoundError(
            f"No FIRMS CSV found in {RAW_DATA_DIR}. "
            "Run data_retrieval.ipynb first so it can save the raw CSV to data/raw/."
        )

raw_df = pd.read_csv(RAW_CSV)

print(f"Project root        : {PROJECT_ROOT}")
print(f"Loaded raw data from: {RAW_CSV}")
print(f"Rows loaded         : {len(raw_df):,}")


Project root        : /Users/davidleu/Desktop/sds210-wildfire-mapping-project
Loaded raw data from: /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/raw/firms_viirs_global_5d.csv
Rows loaded         : 133,061


In [3]:
# ── Quick structural overview ──────────────────────────────────────────────────
print("=" * 60)
print(f"Shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
print("=" * 60)
print("\nColumn names & dtypes:")
print(raw_df.dtypes.to_string())
print("\nMissing values per column:")
print(raw_df.isnull().sum()[raw_df.isnull().sum() > 0].to_string() or "  None")
print("\nFirst three records:")
raw_df.head(3)

Shape: 133,061 rows × 14 columns

Column names & dtypes:
latitude      float64
longitude     float64
bright_ti4    float64
scan          float64
track         float64
acq_date       object
acq_time        int64
satellite      object
instrument     object
confidence     object
version        object
bright_ti5    float64
frp           float64
daynight       object

Missing values per column:
Series([], )

First three records:


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,-10.38626,34.97153,311.74,0.78,0.78,2026-05-07,1,N,VIIRS,n,2.0NRT,285.90,2.92,N
1,-8.20365,32.76895,302.43,0.52,0.67,2026-05-07,1,N,VIIRS,n,2.0NRT,288.00,0.88,N
2,-6.85891,12.39353,319.00,0.37,0.58,2026-05-07,1,N,VIIRS,n,2.0NRT,287.29,8.19,N


In [4]:
# ── Descriptive statistics for key numeric fields ──────────────────────────────
key_cols = ["frp", "bright_ti4", "bright_ti5", "scan", "track"]
# Only show columns that actually exist in this dataset
existing_key_cols = [c for c in key_cols if c in raw_df.columns]
raw_df[existing_key_cols].describe().round(2)

,frp,bright_ti4,bright_ti5,scan,track
count,133061.00,133061.00,133061.00,133061.00,133061.00
mean,7.86,333.27,297.44,0.46,0.49
std,15.48,17.37,10.63,0.09,0.12
min,0.00,208.00,236.86,0.32,0.36
25%,2.29,328.41,290.91,0.39,0.38
50%,4.37,335.95,298.22,0.44,0.45
75%,7.89,343.51,304.22,0.51,0.57
max,594.87,367.00,387.16,0.80,0.78


### 3.1 Cleaning Pipeline

The function below applies all cleaning steps in a single, reproducible pass:

- **Drop rows with missing coordinates or FRP** (the three attributes that are non-negotiable for geospatial mapping)
- **Parse acquisition date and time** into proper `datetime` objects
- **Standardise the confidence field** (VIIRS uses text labels: `'low'`, `'nominal'`, `'high'`)
- **Assign continent labels** via a coarse longitude/latitude bounding-box lookup (for Q2)
- **Add a normalised FRP field** (used to scale marker radii continuously)

In [5]:
def assign_continent(lat: float, lon: float) -> str:
    """
    Return a coarse continent label for a given latitude / longitude pair.

    Uses conservative bounding boxes; points that fall in ambiguous
    border zones are labelled 'Other'.
    """
    # Ordered from smallest to largest bounding box to avoid false matches
    continent_boxes = [
        ("Europe",        35,  72,  -25,  45),
        ("Africa",       -35,  37,  -20,  55),
        ("Asia",          10,  80,   25, 180),
        ("North America", 10,  85, -170, -50),
        ("South America",-60,  15, -120, -30),
        ("Oceania",      -50,   0,  110, 180),
    ]
    for name, lat_min, lat_max, lon_min, lon_max in continent_boxes:
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            return name
    return "Other"


def clean_firms_data(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Apply a reproducible cleaning and enrichment pipeline to raw FIRMS records.

    Steps
    -----
    1. Drop rows missing latitude, longitude, or FRP.
    2. Remove duplicate detections (same lat/lon/date/time).
    3. Parse acq_date and acq_time into a UTC datetime column.
    4. Standardise the 'confidence' column to lowercase text.
    5. Add a 'continent' label.
    6. Add a normalised FRP column (0–1 scale) for marker sizing.

    Parameters
    ----------
    raw : pd.DataFrame — raw FIRMS output.

    Returns
    -------
    pd.DataFrame — cleaned and enriched records.
    """
    df = raw.copy()

    # ── Step 1: Drop rows with critical missing values ─────────────────────────
    required_cols = ["latitude", "longitude", "frp"]
    n_before = len(df)
    df.dropna(subset=required_cols, inplace=True)
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"  Dropped {n_dropped:,} rows with missing lat/lon/frp.")

    # ── Step 2: Remove exact duplicates ───────────────────────────────────────
    dedup_cols = ["latitude", "longitude", "acq_date", "acq_time"]
    dedup_cols = [c for c in dedup_cols if c in df.columns]   # safety check
    n_before = len(df)
    df.drop_duplicates(subset=dedup_cols, inplace=True)
    print(f"  Removed {n_before - len(df):,} duplicate detections.")

    # ── Step 3: Parse acquisition datetime ────────────────────────────────────
    if "acq_date" in df.columns and "acq_time" in df.columns:
        # acq_time is stored as HHMM integer (e.g. 0715, 2345)
        df["acq_time"] = df["acq_time"].astype(str).str.zfill(4)
        df["acq_datetime"] = pd.to_datetime(
            df["acq_date"].astype(str) + " " + df["acq_time"],
            format="%Y-%m-%d %H%M",
            utc=True,
            errors="coerce",
        )

    # ── Step 4: Standardise confidence labels ─────────────────────────────────
    if "confidence" in df.columns:
        df["confidence"] = (
            df["confidence"]
            .astype(str)
            .str.lower()
            .str.strip()
        )
        # Map any numeric confidence (MODIS legacy) to text labels
        confidence_map = {
            str(v): label
            for v_range, label in [
                (range(0,  40), "low"),
                (range(40, 80), "nominal"),
                (range(80, 101), "high"),
            ]
            for v in v_range
        }
        df["confidence"] = df["confidence"].replace(confidence_map)

    # ── Step 5: Assign continent labels ───────────────────────────────────────
    df["continent"] = df.apply(
        lambda row: assign_continent(row["latitude"], row["longitude"]),
        axis=1
    )

    # ── Step 6: Normalised FRP for marker sizing ───────────────────────────────
    frp_max = df["frp"].max()
    frp_min = df["frp"].min()
    if frp_max > frp_min:
        df["frp_normalised"] = (df["frp"] - frp_min) / (frp_max - frp_min)
    else:
        df["frp_normalised"] = 0.5

    df.reset_index(drop=True, inplace=True)
    print(f"  Cleaning complete. Final record count: {len(df):,}")
    return df

In [6]:
print("Applying cleaning pipeline …")
fires_df = clean_firms_data(raw_df)

print("\nCleaned data — first three rows:")
fires_df[["latitude", "longitude", "acq_datetime", "frp",
           "confidence", "continent", "daynight"]].head(3)

Applying cleaning pipeline …
  Removed 0 duplicate detections.
  Cleaning complete. Final record count: 133,061

Cleaned data — first three rows:


,latitude,longitude,acq_datetime,frp,confidence,continent,daynight
0,-10.38626,34.97153,2026-05-07 00:01:00+00:00,2.92,n,Africa,N
1,-8.20365,32.76895,2026-05-07 00:01:00+00:00,0.88,n,Africa,N
2,-6.85891,12.39353,2026-05-07 00:01:00+00:00,8.19,n,Africa,N


### 3.2 Build GeoDataFrame

Converting to a `GeoDataFrame` unlocks GeoPandas spatial operations and enables direct export to geospatial formats.

In [7]:
def build_geodataframe(df: pd.DataFrame, crs: str = "EPSG:4326") -> gpd.GeoDataFrame:
    """
    Convert a cleaned FIRMS DataFrame into a GeoDataFrame.

    Parameters
    ----------
    df  : Cleaned FIRMS records with 'latitude' and 'longitude' columns.
    crs : Coordinate reference system (default WGS 84).

    Returns
    -------
    gpd.GeoDataFrame with a Point geometry column.
    """
    geometry = gpd.points_from_xy(df["longitude"], df["latitude"])
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)
    return gdf


fires_gdf = build_geodataframe(fires_df)
print(f"GeoDataFrame created: {len(fires_gdf):,} fire detections | CRS: {fires_gdf.crs}")

GeoDataFrame created: 133,061 fire detections | CRS: EPSG:4326


### 3.3 Export Cleaned Data

The cleaned tabular and geospatial outputs are saved for later analysis and mapping notebooks.


In [8]:
# Export cleaned outputs for the next notebook / analysis step
CLEAN_CSV = PROCESSED_DATA_DIR / "firms_viirs_cleaned.csv"
CLEAN_GPKG = PROCESSED_DATA_DIR / "firms_viirs_cleaned.gpkg"

fires_df.to_csv(CLEAN_CSV, index=False)
fires_gdf.to_file(CLEAN_GPKG, layer="fires", driver="GPKG")

print(f"Cleaned CSV saved to : {CLEAN_CSV}")
print(f"Cleaned GPKG saved to: {CLEAN_GPKG}")


Cleaned CSV saved to : /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/processed/firms_viirs_cleaned.csv
Cleaned GPKG saved to: /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/processed/firms_viirs_cleaned.gpkg
